In [1]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
import pandas as pd
from langchain_chroma import Chroma
import torch
# print(torch.__version__, torch.version.cuda)
# print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

C:\Users\ankit\AppData\Local\Temp\ipykernel_31540\1007429996.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


CUDA available: True


In [4]:
from langchain_community.document_loaders import TextLoader

# Initialize the loader with the path to your text file
loader = TextLoader("S:/Projects/rag-dissertation/datasets/20 News Group/alt.atheism.txt", encoding="latin-1")

# Load the file into a list of Document objects
docs = loader.load()

# Access the content and metadata
for doc in docs:
    print("--- Metadata ---")
    print(doc.metadata)  # e.g., {'source': 'path/to/your/file.txt'}
    
    print("\n--- Content ---")
    print(doc.page_content)


--- Metadata ---
{'source': 'S:/Projects/rag-dissertation/datasets/20 News Group/alt.atheism.txt'}

--- Content ---
From: mathew <mathew@mantis.co.uk>
Subject: Alt.Atheism FAQ: Atheist Resources

Archive-name: atheism/resources
Alt-atheism-archive-name: resources
Last-modified: 11 December 1992
Version: 1.0

                              Atheist Resources

                      Addresses of Atheist Organizations

                                     USA

FREEDOM FROM RELIGION FOUNDATION

Darwin fish bumper stickers and assorted other atheist paraphernalia are
available from the Freedom From Religion Foundation in the US.

Write to:  FFRF, P.O. Box 750, Madison, WI 53701.
Telephone: (608) 256-8900

EVOLUTION DESIGNS

Evolution Designs sell the "Darwin fish".  It's a fish symbol, like the ones
Christians stick on their cars, but with feet and the word "Darwin" written
inside.  The deluxe moulded 3D plastic fish is $4.95 postpaid in the US.

Write to:  Evolution Designs, 7119 Laurel Canyo

In [ ]:
#loading dataset in langchain

# docs = []
# for col in df.columns:
#     if col == "Unnamed: 0":
#         continue
#     text = "\n".join(df[col].dropna().astype(str))
#     docs.append(Document(page_content=text, metadata={"column": col}))

# print(len(docs))   # = number of columns
# print(docs[0].metadata)
# print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [5]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

6530
PHILIP K. DICK

Philip K. Dick Dick wrote many philosophical and thought-provoking short 
stories and novels.  His stories are bizarre at times, but very approachable.
He wrote mainly SF, but he wrote about people, truth and religion rather than
technology.  Although he often believed that he had met some sort of God, he
remained sceptical.  Amongst his novels, the following are of some relevance:

"Galactic Pot-Healer"
A fallible alien deity summons a group of Earth craftsmen and women to a
remote planet to raise a giant cathedral from beneath the oceans.  When the
deity begins to demand faith from the earthers, pot-healer Joe Fernwright is
unable to comply.  A polished, ironic and amusing novel.

"A Maze of Death"
Noteworthy for its description of a technology-based religion.


In [6]:
#Embeddings Model 

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"batch_size": 512},
    )

embeddings._client.half()                 # fp16
embeddings._client.max_seq_length = 256   # truncate long abstracts

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

print(type(embeddings))
print(hasattr(embeddings, "_client"), hasattr(embeddings, "client"))

s:\Projects\rag-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 661.02it/s]


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True False


In [7]:
m = embeddings._client
print(m.max_seq_length)                  # 256
print(next(m.parameters()).dtype)        # torch.float16
print(next(m.parameters()).device)       # cuda:0

256
torch.float16
cuda:0


In [7]:
# Embeddings & vector storage
#
# This cell only READS the Chroma DB that was persisted to ./chroma_langchain_db
# (src/chroma_langchain_db) -- it does not re-embed the dataset on every run.

from pathlib import Path

CHROMA_DIR = Path("chroma_langchain_db")
if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f"Local Chroma DB not found at {CHROMA_DIR.resolve()}. "
        "Run the ingestion cells once with REBUILD_INDEX = True to create it."
    )

vector_store = Chroma(
    collection_name="General_Embeddings",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

# Default LangChain retriever over the persisted collection.
retriever = vector_store.as_retriever()

print(f"Loaded {vector_store._collection.count()} chunks from {CHROMA_DIR.resolve()}")


Loaded 0 chunks from S:\Projects\rag-dissertation\src\chroma_langchain_db


In [8]:
# Timeline of storage
#
# The index already exists on disk, so ingestion is opt-in: flip REBUILD_INDEX to
# True only when you want to (re)embed the split texts into the collection.

from tqdm import tqdm

REBUILD_INDEX = True

if REBUILD_INDEX:
    max_bs = vector_store._client.get_max_batch_size()   # 5461 for you
    BATCH = min(2000, max_bs)                            # stay safely under it

    for i in tqdm(range(0, len(texts), BATCH)):
        batch = texts[i:i + BATCH]
        ids = [str(d.metadata.get("pmid", i + j)) for j, d in enumerate(batch)]
        vector_store.add_documents(batch, ids=ids)
else:
    print("Skipping ingestion; reading the existing local Chroma DB as-is.")


100%|██████████| 4/4 [16:40<00:00, 250.12s/it]


In [9]:
# Retrieve evidence for the same question that will be evaluated.
query = "What did Robert Beauchaine write about god?"
retrieved_docs = retriever.invoke(query)
retrieval_context = [doc.page_content for doc in retrieved_docs]
if not retrieval_context:
    raise ValueError("The retriever returned no context; cannot evaluate a RAG answer.")
print(f"Retrieved {len(retrieval_context)} context chunks.")

# Show the retrieved chunks (rank, metadata, text) so the evidence is visible.
for rank, doc in enumerate(retrieved_docs, start=1):
    meta = ", ".join(f"{k}={v}" for k, v in doc.metadata.items()) or "none"
    print(f"--- chunk {rank}/{len(retrieved_docs)} [{meta}] ---")
    print(doc.page_content)
    print()


Retrieved 4 context chunks.
--- chunk 1/4 [source=S:/Projects/rag-dissertation/datasets/20 News Group/alt.atheism.txt] ---
Described by 'Rolling Stone' as "A sick masterpiece for those who can still
laugh at the fact that nothing is funny anymore."  The official Bible of the
SubGenius Church, containing the sacred teachings of J.R. "Bob" Dobbs. 
Instant answers to everything; causes catalytic brain cell loss in seconds;
the secret of total slack; how to relax in the safety of your delusions and
pull the wool over your own eyes; nuclear doom and other things to laugh at.

QUOTE:

"He has been known to answer questions concerning universal truths with 
screams.  With suggestive silence.  By peeing down his pants leg.  His most
famous sermon was of cosmic simplicity: "Bob" standing on the stage with his
hands in his pockets, smoking, looking around and saying nothing.  Heated
arguments still rage among the monks, often erupting into fatal duels, as
towhether the Master consulted his wrist

In [10]:
#Setting up LLM

llm = ChatOllama(
    model="qwen3.5:2b", 
    temperature = 0.5,
    reasoning=False,
    
)

In [11]:
#Pre Retrieval - Baseline
# context_text = "\n\n".join(retrieval_context)

response = llm.invoke(query)
print(response.content)


Robert Beauchaine (1745–1806), an English clergyman and writer, is best known for his work **"The Doctrine of the Trinity"**, published in 1791. In this text, he sought to provide a clear, rational explanation of the Christian doctrine of the Trinity based on Scripture and reason, aiming to combat what he viewed as vague or heretical interpretations of the concept.

Beauchaine approached the subject with a blend of theological rigor and philosophical clarity. He argued that the doctrine cannot be understood through abstract speculation alone but must be grounded in biblical texts and the nature of God's unity and diversity within the person of the Holy Spirit. His work is particularly noted for its attempt to make complex Trinitarian theology accessible to laypeople while maintaining strict adherence to orthodox Christian dogma.

However, Beauchaine's influence was limited by his time and the prevailing religious climate of early 19th-century England. He did not achieve widespread popu

In [12]:
#Pre Retrieval - Baseline Eval

import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    """Newsgroup: alt.atheism
document_id: 53261
From: dps@nasa.kodak.com (Dan Schaertel,,,)
Subject: Re: Christian Morality is

In article 11853@vice.ICO.TEK.COM, bobbe@vice.ICO.TEK.COM (Robert Beauchaine) writes:
|>
|>  Yet I am still not a believer.  Is god not concerned with my
|>  disposition?  Why is it beneath him to provide me with the
|>  evidence I would require to believe?  The evidence that my
|>  personality, given to me by this god, would find compelling?

The fact is God could cause you to believe anything He wants you to. 
But think about it for a minute.  Would you rather have someone love
you because you made them love you, or because they wanted to
love you.     The responsibility is on you to love God and take a step toward
Him.  He promises to be there for you, but you have to look for yourself.
Those who doubt this or dispute it have not givin it a sincere effort.
Simple logic arguments are folly.  If you read the Bible you will see
that Jesus made fools of those who tried to trick him with "logic".
Our ability to reason is just a spec of creation.  Yet some think it is
the ultimate.  If you rely simply on your reason then you will never
know more than you do now.   To learn you must accept that which
you don't know."""

]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="gemma4:e4b",
    base_url="http://localhost:11434",
    temperature=0,
    # reasoning=False,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\ankit\AppData\Local\Temp\ipykernel_31540\981561716.py:90: DeprecationWarning: 'HallucinationMetric' now scores in the same direction as every other deepeval metric: 1 is a pass, 0 is a failure, and 'threshold' is the MINIMUM passing score. It previously scored the proportion of violations, where 'threshold' was a maximum. Review any 'threshold' you pass and any code reading '.score' - a threshold of 0.2 that used to mean 'at most 20% violations' should now be 0.8. This notice will be removed in a future release.
  "hallucination": HallucinationMetric(


hallucination: 0.000
  passed: False
  reason: The score is 0.00 because the actual output introduces information about Robert Beauchaine's work on the Trinity, which is entirely absent from the provided context (an email exchange about faith, doubt, and reason).


faithfulness: 0.000
  passed: False
  reason: The score is 0.00 because the actual output incorrectly claims that 'Robert Beauchaine wrote about God primarily through his work *The Doctrine of the Trinity* (1791)' and discusses specific theological arguments, which contradicts the context by mentioning an email from Bob Beauchaine (bobbe@vice.ICO.TEK.COM) but providing no information about 'Robert Beauchaine (1745–1806)' or any work titled 'The Doctrine of the Trinity' published in 1791.


answer relevancy: 0.800
  passed: True
  reason: The score is 0.80 because while the output correctly identifies that Beauchaine wrote about God, several statements included biographical details (life dates) or discussed secondary aspects like critical reception or the limited influence of his work, which are tangential and prevent a perfect score.


In [13]:
# Post Retrieval Query
context_text = "\n\n".join(retrieval_context)
prompt = f"""Answer the question using only the supplied context. If the context does not contain the answer, say so.

Context:
{context_text}

Question: {query}
Answer:"""
response = llm.invoke(prompt)
print(response.content)


Based on the provided context, there is no information regarding what Robert Beauchaine wrote specifically about God. The text mentions that he was described by *Rolling Stone* as "A sick masterpiece for those who can still laugh at the fact that nothing is funny anymore," identifies him as J.R. "Bob" Dobbs, and notes he is the official Bible of the SubGenius Church containing teachings on topics like nuclear doom and brain cell loss, but it does not detail his writings or specific content regarding God.


In [14]:
#Post Retrieval Eval
import os

# The local judge can be slow, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

actual_context =[
    """Newsgroup: alt.atheism
    document_id: 53261
    From: dps@nasa.kodak.com (Dan Schaertel,,,)
    Subject: Re: Christian Morality is
    
    In article 11853@vice.ICO.TEK.COM, bobbe@vice.ICO.TEK.COM (Robert Beauchaine) writes:
    |>
    |>  Yet I am still not a believer.  Is god not concerned with my
    |>  disposition?  Why is it beneath him to provide me with the
    |>  evidence I would require to believe?  The evidence that my
    |>  personality, given to me by this god, would find compelling?
    
    The fact is God could cause you to believe anything He wants you to. 
    But think about it for a minute.  Would you rather have someone love
    you because you made them love you, or because they wanted to
    love you.     The responsibility is on you to love God and take a step toward
    Him.  He promises to be there for you, but you have to look for yourself.
    Those who doubt this or dispute it have not givin it a sincere effort.
    Simple logic arguments are folly.  If you read the Bible you will see
    that Jesus made fools of those who tried to trick him with "logic".
    Our ability to reason is just a spec of creation.  Yet some think it is
    the ultimate.  If you rely simply on your reason then you will never
    know more than you do now.   To learn you must accept that which
    you don't know."""
]

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models.llms.ollama_model import OllamaModel, retry_ollama
from deepeval.test_case import Audio, LLMTestCase
from pydantic import BaseModel
from typing import List, Optional, Tuple, Union


class NonThinkingOllamaModel(OllamaModel):
    @retry_ollama
    def generate(
        self,
        prompt: str,
        schema: Optional[BaseModel] = None,
        audio: Optional[List[Audio]] = None,
    ) -> Tuple[Union[str, BaseModel], float]:
        response = self.load_model().chat(
            model=self.name,
            messages=self._prompt_messages(prompt, audio),
            think=False,
            format=schema.model_json_schema() if schema else None,
            options={
                "temperature": self.temperature,
                **self.generation_kwargs,
            },
        )
        content = response.message.content
        if not content:
            raise ValueError(
                "Ollama returned empty content for the DeepEval judge response."
            )
        result = schema.model_validate_json(content) if schema else content
        return result, 0

# Use a separate local judge; Qwen 3.5 otherwise returns reasoning instead of JSON.
evaluation_model = NonThinkingOllamaModel(
    model="gemma4:e4b",
    base_url="http://localhost:11434",
    temperature=0,
    # reasoning=False,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=actual_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

hallucination: 0.000
  passed: False
  reason: The score is 0.00 because the actual output introduces multiple claims (mentioning *Rolling Stone*, J.R. "Bob" Dobbs, and the SubGenius Church) that are entirely unsupported and absent from the provided context, which only contains a quote from Robert Beauchaine.


faithfulness: 0.250
  passed: False
  reason: The score is 0.25 because the actual output contradicts the context in several ways: it incorrectly claims the email from Bob Beauchaine references a quote from *Rolling Stone* about his work, when the context only mentions the email and the article source; it incorrectly states that Bob Beauchaine is J.R. "Bob" Dobbs, despite the context only mentioning that Kent Sandvik wrote an article; and it incorrectly details the content of the official Bible of the SubGenius Church regarding God, when the context only mentions topics like nuclear doom and brain cell loss.


answer relevancy: 0.500
  passed: True
  reason: The score is 0.50 because while the output attempts to answer the question, several statements are irrelevant. Specifically, the output includes general quotes about his work, descriptions of church affiliations, and general church teachings, none of which directly address what Robert Beauchaine wrote *about God* as requested.
